In [29]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [30]:
origem = [1,2,3]
oferta = [30,40,20]

destino = [4,5,6]
demanda = [10,30,40]

no_intermediario = [4,5]

caminhos = [(1,3),(1,4),(2,4),(2,5),(3,4),(3,6),(4,6),(5,4),(5,6)]
custos = [20,30,10,20,20,40,30,50,30]


In [31]:
#MODELO

model = pyo.ConcreteModel()

model.origem = pyo.Set(initialize=origem)
model.destino = pyo.Set(initialize=destino)
model.oferta = pyo.Param(model.origem, initialize=dict(zip(origem, oferta)))
model.demanda = pyo.Param(model.destino, initialize=dict(zip(destino, demanda)))
model.caminhos = pyo.Set(initialize=caminhos, dimen=2)
model.custos = pyo.Param(model.caminhos, initialize=dict(zip(caminhos, custos)))
model.x = pyo.Var(model.caminhos, domain=pyo.NonNegativeIntegers)

In [32]:
#objetivo

def objetivo(model):
    return sum(model.custos[i,j]*model.x[i,j] for (i,j) in model.caminhos)
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.minimize)

In [36]:
#restrições

def restricao_oferta(model, o):
    saida  = sum(model.x[c] for c in model.caminhos if c[0] == o)
    entrada = sum(model.x[c] for c in model.caminhos if c[1] == o)
    return saida - entrada <= model.oferta[o]

model.restricao_oferta = pyo.Constraint(model.origem, rule=restricao_oferta)

def restricao_demanda(model, j):
    entrada = sum(model.x[c] for c in model.caminhos if c[1] == j)
    saida   = sum(model.x[c] for c in model.caminhos if c[0] == j)
    return entrada - saida >= model.demanda[j]

model.restricao_demanda = pyo.Constraint(model.destino, rule=restricao_demanda)

(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().


In [43]:
model.pprint()

3 Set Declarations
    caminhos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    9 : {(1, 3), (1, 4), (2, 4), (2, 5), (3, 4), (3, 6), (4, 6), (5, 4), (5, 6)}
    destino : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {4, 5, 6}
    origem : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {1, 2, 3}

3 Param Declarations
    custos : Size=9, Index=caminhos, Domain=Any, Default=None, Mutable=False
        Key    : Value
        (1, 3) :    20
        (1, 4) :    30
        (2, 4) :    10
        (2, 5) :    20
        (3, 4) :    20
        (3, 6) :    40
        (4, 6) :    30
        (5, 4) :    50
        (5, 6) :    30
    demanda : Size=3, Index=destino, Domain=Any, Default=None, Mutable=False
        Key : Value
          4 :    10
          5 :    30
        

In [40]:
opt = pyo.SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp7cmrd7aw.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpul8h0by8.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpul8h0by8.pyomo.lp
Objective sense      : Minimize
Variables            :       9  [General Integer: 9]
Objective nonzeros   :       9
Linear constraints   :       6  [Less: 3,  Greater: 3]
  Nonzeros           :      18
  RHS nonzeros       :       6

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 10

In [44]:
for i in model.caminhos:
    print(f"x[{i}] = {model.x[i].value},custo total = {model.x[i].value * model.custos[i]}    ")

print(f"Custo total = {pyo.value(model.objetivo)}")

x[(1, 3)] = 20.0,custo total = 400.0    
x[(1, 4)] = 0.0,custo total = 0.0    
x[(2, 4)] = 10.0,custo total = 100.0    
x[(2, 5)] = 30.0,custo total = 600.0    
x[(3, 4)] = 0.0,custo total = 0.0    
x[(3, 6)] = 40.0,custo total = 1600.0    
x[(4, 6)] = 0.0,custo total = 0.0    
x[(5, 4)] = 0.0,custo total = 0.0    
x[(5, 6)] = 0.0,custo total = 0.0    
Custo total = 2700.0
